# Notebook 03: Natural Language Processing - Text Summarization

**Objective:** Learn how to automatically condense long documents into concise summaries using sequence-to-sequence transformer models.

**Prerequisites:** Python basics. Notebook 01 (Text Generation) is helpful for understanding generation parameters.

**Learning Objectives:**
- Understand the difference between extractive and abstractive summarization
- Use the Pipeline API and direct model loading for summarization
- Control summary length and quality with generation parameters
- Apply summarization to real-world document types
- Identify when summarization models produce unreliable output

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min RAM | Recommended Setup | Notes |
|--------------|------------|------|---------|-------------------|-------|
| **CPU (Small)** | sshleifer/distilbart-cnn-12-6 | 1.2GB | 4GB | 4GB RAM, CPU | Faster inference |
| **GPU (Medium)** | facebook/bart-large-cnn | 1.6GB | 6GB | 8GB VRAM (RTX 4080) | Better quality |

### Software Requirements
- Python 3.10+
- Libraries: `transformers`, `torch`, `pandas`
- See `requirements.txt` for full list

## Overview

**Text summarization** condenses long documents into shorter versions while preserving the key information. There are two main approaches:

- **Extractive**: Selects the most important *existing* sentences from the source text and concatenates them. Think of it as highlighting key passages.
- **Abstractive**: Generates *new* sentences that capture the meaning of the original text. This is what we'll focus on, since it produces more natural, human-like summaries.

**How abstractive summarization works:**
1. The **encoder** reads the entire input document and produces a representation
2. The **decoder** generates the summary token-by-token, attending to the encoder's representation
3. The model was trained on pairs of (full article, human-written summary)

This is a **sequence-to-sequence** (seq2seq) architecture, fundamentally different from the encoder-only (BERT, Notebook 02) and decoder-only (GPT-2, Notebook 01) models we've seen so far. BART combines both an encoder *and* a decoder.

**Use cases:** News article summaries, meeting notes, research paper abstracts, document digests, email summarization.

## Expected Behaviors

### First Time Running
- **Model download**: ~1.2GB for DistilBART (3-5 minutes)
- Largest model so far in the NLP notebooks
- Cached in `~/.cache/huggingface/hub/` for subsequent runs

### Summary Quality
- Summaries are **abstractive** -- the model generates new sentences, not just extracts
- Typically 20-30% of original text length
- Works best with factual, structured text (news articles, reports)

### Performance
- **Short text (100 words)**: CPU 2-4s, GPU 0.5-1s
- **Medium text (300 words)**: CPU 5-8s, GPU 1-2s
- **Long text (500+ words)**: CPU 10-15s, GPU 2-4s

### Beam Search Impact
- `num_beams=1`: Fastest, lower quality
- `num_beams=4`: Good balance (default)
- `num_beams=8`: Best quality, ~2x slower

## Setup and Installation

All imports, seeds, and environment checks in a single cell.

In [ ]:
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline, set_seed

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Version metadata
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Model Selection

**DistilBART** is a distilled version of BART-large-cnn, trained to produce summaries of CNN/DailyMail news articles. It retains most of BART's quality while being ~2x faster. Both models have the same encoder-decoder architecture; the difference is the number of layers (6 decoder layers in DistilBART vs 12 in BART-large).

In [ ]:
# CHOOSE YOUR MODEL:

# Option 1: CPU-friendly (recommended for beginners)
MODEL_NAME = "sshleifer/distilbart-cnn-12-6"  # 1.2GB, distilled BART

# Option 2: GPU-optimized (uncomment if you have RTX 4080 or similar)
# MODEL_NAME = "facebook/bart-large-cnn"  # 1.6GB, better quality

print(f"Selected model: {MODEL_NAME}")

---

## Method 1: Using Pipeline (Simplest)

The summarization pipeline works just like the text generation and classification pipelines from Notebooks 01 and 02. You pass text in, and get a summary out. The main new parameters are `min_length` and `max_length`, which control the summary bounds.

In [ ]:
try:
    print(f"Loading {MODEL_NAME}...")
    summarizer = pipeline(
        "summarization",
        model=MODEL_NAME,
        device=0 if torch.cuda.is_available() else -1,
    )
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    print("This model is ~1.2GB. Check your internet connection.")
    raise

### Basic Summarization

Let's summarize a paragraph about AI. The `max_length` and `min_length` parameters set bounds on the summary length in tokens (roughly words).

In [ ]:
article = """
The field of artificial intelligence has seen remarkable progress in recent years, 
particularly in the domain of natural language processing. Large language models, 
which are trained on vast amounts of text data, have demonstrated impressive capabilities 
in understanding and generating human-like text. These models use transformer architectures, 
which employ attention mechanisms to process sequential data more effectively than previous 
approaches. The transformer architecture was introduced in the landmark paper "Attention Is All 
You Need" in 2017. Since then, models like BERT, GPT, and their variants have revolutionized 
numerous NLP tasks including translation, summarization, question answering, and text generation. 
However, these models also raise important concerns about computational costs, environmental impact, 
and potential misuse. Researchers are now focusing on making these models more efficient, 
interpretable, and aligned with human values.
"""

summary = summarizer(article, max_length=130, min_length=30, do_sample=False)

original_words = len(article.split())
summary_words = len(summary[0]["summary_text"].split())

print(f"Original: {original_words} words")
print(f"Summary:  {summary_words} words ({summary_words/original_words:.0%} of original)\n")
print(summary[0]["summary_text"])

Notice the summary is **abstractive** -- it rephrases the content rather than copying sentences verbatim. This is BART's strength: it produces fluent, natural summaries.

### Controlling Summary Length

The `max_length` and `min_length` parameters let you control how concise or detailed the summary should be. Let's compare different length settings on the same input.

In [ ]:
def compare_summary_lengths(
    text: str,
    length_configs: list[tuple[int, int]] | None = None,
) -> pd.DataFrame:
    """Generate summaries at different length settings and compare.

    Args:
        text: Source text to summarize.
        length_configs: List of (max_length, min_length) tuples.
            Defaults to [(50, 20), (100, 40), (150, 60)].

    Returns:
        DataFrame with length setting, word count, and summary text.
    """
    if length_configs is None:
        length_configs = [(50, 20), (100, 40), (150, 60)]

    rows: list[dict] = []
    for max_len, min_len in length_configs:
        result = summarizer(text, max_length=max_len, min_length=min_len, do_sample=False)
        summary_text = result[0]["summary_text"]
        rows.append({
            "Length Setting": f"{min_len}-{max_len} tokens",
            "Words": len(summary_text.split()),
            "Summary": summary_text,
        })

    return pd.DataFrame(rows)


length_df = compare_summary_lengths(article)
length_df

Shorter summaries are more aggressive at discarding details, keeping only the core message. Longer summaries preserve more nuance. The right length depends on your use case.

---

## Method 2: Using Model and Tokenizer Directly (Advanced)

Loading the model directly gives access to beam search parameters and other generation controls. This is useful when you need to tune quality vs. speed tradeoffs, or when integrating summarization into a larger pipeline.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.to(device)

print(f"Model loaded on: {device}")
print(f"Encoder layers: {model.config.encoder_layers}")
print(f"Decoder layers: {model.config.decoder_layers}")
print(f"Max input length: {tokenizer.model_max_length} tokens")

With direct access to `model.generate()`, we can control **beam search** -- the algorithm that explores multiple possible summaries simultaneously and picks the best one. More beams mean higher quality but slower generation.

The `length_penalty` parameter biases the model toward longer (>1.0) or shorter (<1.0) summaries.

In [ ]:
def summarize_with_beams(
    text: str,
    max_length: int = 100,
    min_length: int = 30,
    num_beams: int = 4,
    length_penalty: float = 2.0,
) -> str:
    """Summarize text using direct model inference with beam search.

    Args:
        text: Source text to summarize.
        max_length: Maximum summary length in tokens.
        min_length: Minimum summary length in tokens.
        num_beams: Number of beams for beam search (higher = better quality, slower).
        length_penalty: Values > 1.0 encourage longer summaries.

    Returns:
        The generated summary string.
    """
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True).to(device)

    summary_ids = model.generate(
        inputs.input_ids,
        max_length=max_length,
        min_length=min_length,
        num_beams=num_beams,
        length_penalty=length_penalty,
        early_stopping=True,
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


climate_text = """
Climate change is one of the most pressing challenges facing humanity today. 
Rising global temperatures are causing ice caps to melt, sea levels to rise, 
and extreme weather events to become more frequent. Scientists warn that without 
immediate action to reduce greenhouse gas emissions, the consequences could be 
catastrophic. Renewable energy sources like solar and wind power offer promising 
solutions, but transitioning away from fossil fuels requires coordinated global 
effort and significant investment. Many countries have committed to achieving 
net-zero emissions by 2050, but current progress suggests more ambitious actions 
are needed.
"""

summary = summarize_with_beams(climate_text)
print("Summary:", summary)

### Comparing Beam Search Settings

Let's see how the number of beams affects both quality and speed.

In [ ]:
def compare_beam_settings(
    text: str,
    beam_counts: list[int] | None = None,
) -> pd.DataFrame:
    """Compare summarization quality and speed across beam counts.

    Args:
        text: Source text to summarize.
        beam_counts: List of beam counts to test. Defaults to [1, 2, 4, 8].

    Returns:
        DataFrame with beam count, time, and generated summary.
    """
    if beam_counts is None:
        beam_counts = [1, 2, 4, 8]

    rows: list[dict] = []
    for beams in beam_counts:
        start = time.time()
        summary = summarize_with_beams(text, num_beams=beams)
        elapsed = time.time() - start

        rows.append({
            "Beams": beams,
            "Time (s)": round(elapsed, 2),
            "Words": len(summary.split()),
            "Summary": summary[:80] + "..." if len(summary) > 80 else summary,
        })

    return pd.DataFrame(rows)


print(f"Comparing beam search on {device}...\n")
beam_df = compare_beam_settings(article)
beam_df

More beams produce slightly better summaries but at increasing time cost. For most applications, `num_beams=4` is the sweet spot. `num_beams=1` (greedy decoding) is fastest but may produce less fluent text.

---

## Practical Applications

Let's apply summarization to common document types. Each example shows a different use pattern and text style.

### Example 1: News Article Summarization

News articles are the ideal input for BART-based summarizers, since the model was specifically trained on CNN/DailyMail news.

In [ ]:
news_article = """
Tech giant announces breakthrough in quantum computing. The company revealed a new 
quantum processor that achieved quantum supremacy, solving complex problems that would 
take classical computers thousands of years to complete. The 70-qubit processor 
demonstrated error rates low enough for practical applications. Industry experts believe 
this development could revolutionize fields like drug discovery, financial modeling, and 
cryptography. However, practical quantum computers for everyday use are still years away. 
The company plans to make the technology available to researchers through cloud access 
next year. Competitors are racing to develop their own quantum systems, with several 
other tech companies announcing similar initiatives. The quantum computing market is 
expected to grow to $65 billion by 2030 according to industry analysts.
"""

summary = summarizer(news_article, max_length=60, min_length=25, do_sample=False)
print(f"Original: {len(news_article.split())} words")
print(f"Summary:  {len(summary[0]['summary_text'].split())} words\n")
print(summary[0]["summary_text"])

### Example 2: Meeting Notes Summarization

Meeting notes have a different structure from news articles -- they contain action items, names, and dates. Let's see how the model handles this.

In [ ]:
meeting_notes = """
The quarterly team meeting began with a review of the project timeline. Sarah reported 
that the mobile app development is on track for the March release. However, backend 
integration is facing delays due to API compatibility issues. Mike volunteered to work 
with the infrastructure team to resolve these problems by next week. The marketing team 
presented their campaign strategy, proposing a budget increase of 15% for social media 
advertising. After discussion, the team agreed to the proposal pending approval from 
upper management. Action items include: Sarah to complete user testing by Friday, 
Mike to fix API issues by next Tuesday, and Jennifer to submit the budget proposal 
by end of week. The next meeting is scheduled for two weeks from today.
"""

summary = summarizer(meeting_notes, max_length=80, min_length=30, do_sample=False)
print(f"Original: {len(meeting_notes.split())} words")
print(f"Summary:  {len(summary[0]['summary_text'].split())} words\n")
print(summary[0]["summary_text"])

### Example 3: Batch Summarization

Just like classification, we can summarize multiple documents in a batch. Let's process three documents and present the results in a table.

In [ ]:
def summarize_batch(
    documents: list[str],
    max_length: int = 50,
    min_length: int = 20,
) -> pd.DataFrame:
    """Summarize multiple documents and present results as a DataFrame.

    Args:
        documents: List of source texts to summarize.
        max_length: Maximum summary length in tokens.
        min_length: Minimum summary length in tokens.

    Returns:
        DataFrame with document index, original length, summary length, and summary.
    """
    summaries = summarizer(documents, max_length=max_length, min_length=min_length, do_sample=False)

    rows: list[dict] = []
    for i, (doc, summ) in enumerate(zip(documents, summaries), 1):
        summary_text = summ["summary_text"]
        rows.append({
            "Doc": i,
            "Original Words": len(doc.split()),
            "Summary Words": len(summary_text.split()),
            "Compression": f"{len(summary_text.split()) / len(doc.split()):.0%}",
            "Summary": summary_text,
        })

    return pd.DataFrame(rows)


documents = [
    "A new study shows that regular exercise can significantly improve cognitive function "
    "in older adults. Researchers followed 500 participants over three years, finding that "
    "those who engaged in moderate aerobic exercise for 30 minutes daily showed 25% better "
    "memory retention than the control group.",

    "Electric vehicle sales reached a record high this quarter, with major manufacturers "
    "reporting 40% year-over-year growth. The surge is attributed to improved battery "
    "technology, expanded charging infrastructure, and government incentives. Industry "
    "analysts predict EVs will account for 50% of new car sales by 2030.",

    "A recent archaeological discovery in Egypt has uncovered a previously unknown pharaoh's "
    "tomb. The tomb, dating back 3,400 years, contains remarkably well-preserved artifacts "
    "and hieroglyphics that provide new insights into ancient Egyptian society. Experts "
    "believe this finding could rewrite parts of Egyptian history.",
]

batch_df = summarize_batch(documents)
batch_df

---

## Performance Benchmarking

Summarization is significantly slower than classification because the model must generate tokens one by one (autoregressive decoding). Let's measure how speed scales with input length.

In [ ]:
def benchmark_summarization(
    num_runs: int = 2,
) -> pd.DataFrame:
    """Benchmark summarization speed across different input lengths.

    Args:
        num_runs: Number of runs to average per input length.

    Returns:
        DataFrame with input length, output length, time, and throughput.
    """
    # Create inputs of different lengths by repeating the base article
    base = article.strip()
    test_inputs = [
        ("Short (~50 words)", base[:250]),
        ("Medium (~120 words)", base),
        ("Long (~250 words)", base + " " + base),
    ]

    rows: list[dict] = []
    for label, text in test_inputs:
        times: list[float] = []
        for _ in range(num_runs):
            start = time.time()
            summarizer(text, max_length=60, min_length=20, do_sample=False)
            times.append(time.time() - start)

        avg_time = np.mean(times)
        rows.append({
            "Input": label,
            "Input Words": len(text.split()),
            "Avg Time (s)": round(avg_time, 2),
            "Device": "GPU" if torch.cuda.is_available() else "CPU",
        })

    return pd.DataFrame(rows)


print(f"Benchmarking {MODEL_NAME} on {device}...\n")
bench_df = benchmark_summarization()
bench_df

Summarization time grows with input length because the encoder must process more tokens. The decoder time is relatively fixed (bounded by `max_length`). GPU acceleration helps most on longer inputs.

---

## Error Analysis: Where Summarization Fails

Summarization models have specific failure modes that are important to understand before deploying them. Let's test texts that are designed to expose these weaknesses.

In [ ]:
def analyze_failure_modes(
    test_cases: list[tuple[str, str]],
    max_length: int = 60,
) -> pd.DataFrame:
    """Test summarization on cases designed to expose failure modes.

    Args:
        test_cases: List of (text, category) tuples.
        max_length: Maximum summary length.

    Returns:
        DataFrame showing category, input excerpt, and generated summary.
    """
    rows: list[dict] = []
    for text, category in test_cases:
        result = summarizer(text, max_length=max_length, min_length=10, do_sample=False)
        rows.append({
            "Category": category,
            "Input (excerpt)": text[:60] + "...",
            "Summary": result[0]["summary_text"],
        })
    return pd.DataFrame(rows)


test_cases = [
    # Very short input -- not enough content to summarize
    ("The cat sat on the mat. It was a sunny day.", "Too short"),

    # Narrative / creative text -- model trained on news, not fiction
    ("She opened the door slowly, her heart pounding. The room was dark except for a "
     "single candle flickering on the table. A note lay beside it, written in handwriting "
     "she didn't recognize. 'Meet me at midnight,' it read. She looked at the clock. "
     "It was 11:45.", "Creative fiction"),

    # Numeric / tabular data -- summaries lose precision
    ("Q1 revenue was $12.3M (up 15%), Q2 was $14.1M (up 22%), Q3 was $13.8M (down 2%), "
     "and Q4 was $16.7M (up 21%). Total annual revenue reached $56.9M, representing 14% "
     "year-over-year growth. Operating margin improved from 18% to 22%.", "Numeric data"),

    # Contradictory information
    ("The company reported strong earnings this quarter, beating analyst expectations by 20%. "
     "However, the CEO warned that next quarter would see significant losses due to supply "
     "chain disruptions. The stock price fell 15% despite the positive earnings report, "
     "reflecting investor concern about the future outlook.", "Contradictory"),
]

print("Failure Mode Analysis:\n")
failure_df = analyze_failure_modes(test_cases)
failure_df

**What to notice:**

- **Too short**: When the input is already brief, the "summary" may be longer than the original or just rephrase it awkwardly. Summarization is most useful on texts of 100+ words.
- **Creative fiction**: The model struggles with narrative text because it was trained on news articles. It may flatten the story into a factual-sounding report.
- **Numeric data**: The model often drops or rounds specific numbers. If exact figures matter, summarization models are unreliable.
- **Contradictory information**: The model may emphasize one side of the contradiction and ignore the other, giving a misleading summary.

**Best practices:** Always review automated summaries, especially for high-stakes content. Consider using summaries as drafts for human editors rather than as final output.

---

## Exercises

1. **Length experiment**: Use `compare_summary_lengths()` with configs `[(30, 10), (60, 20), (120, 40), (200, 60)]`. At what point does more length stop adding useful information?

2. **Custom articles**: Find a news article online, paste it in, and summarize it. How does quality compare to the examples above?

3. **Beam search trade-off**: Use `compare_beam_settings()` with `[1, 2, 4, 8, 16]` beams. Plot or print the time vs. beam count. Is 16 beams worth the cost?

4. **Model comparison**: If you have GPU, compare `distilbart-cnn-12-6` with `bart-large-cnn` on the same text. How much better is the larger model?

5. **Long document**: Create a text of 500+ words (concatenate multiple paragraphs) and summarize it. Does the model handle it well, or does it lose important details?

---

## State-of-the-Art Open Models (Reference)

| Model | Developer | Size | Speed | Quality | Best For |
|-------|-----------|------|-------|---------|----------|
| **DistilBART** | HuggingFace | 1.2GB | Fast | Good | Learning, CPU inference |
| **BART-large** | Meta | 1.6GB | Medium | Great | General summarization |
| **PEGASUS** | Google | 2.3GB | Slow | Excellent | News articles |
| **T5-Large** | Google | 3GB | Slow | Excellent | Diverse tasks |
| **LED** | Allen AI | 1.6GB | Very Slow | Great | Long documents (10K+ words) |
| **LongT5** | Google | 3GB | Very Slow | Excellent | Very long documents |

**ROUGE-2 scores (CNN/DailyMail):** DistilBART ~19.5, BART-large ~21.3, PEGASUS ~21.9, T5-3B ~22.5

For long documents (research papers, legal texts), **LED** and **LongT5** are essential -- standard models truncate input at 1024 tokens.

---

## Key Takeaways

- **Abstractive summarization** generates new sentences that capture the meaning, unlike extractive methods that copy sentences
- **Length parameters** (`max_length`, `min_length`) and **beam search** (`num_beams`) are the main quality/speed controls
- **Seq2seq architecture** (encoder + decoder) is fundamentally different from encoder-only (BERT) and decoder-only (GPT-2) models
- Summarization works best on **factual, structured text** and can fail on short inputs, fiction, numeric data, and contradictions
- **Batch processing** is efficient for summarizing multiple documents

## Next Steps

- **Notebook 04 (CV): Image Classification** -- Apply transformers to computer vision
- **Notebook 04-05 (NLP): Fine-tuning** -- Train your own summarization model
- [HuggingFace Summarization Models](https://huggingface.co/models?pipeline_tag=summarization)

## Resources

- [Summarization Task Guide](https://huggingface.co/docs/transformers/tasks/summarization)
- [BART Paper](https://arxiv.org/abs/1910.13461)
- [ROUGE Metric](https://huggingface.co/spaces/evaluate-metric/rouge)